# Example: Static inverse free-boundary equilibrium calculations using JAX

---

This example notebook shows how to use the JAX solver in FreeGSNKE to solve **static inverse** free-boundary Grad-Shafranov (GS) problems.

In the **inverse** solve mode we solve for the active coil currents that give a desired equilibrium shape using a user-defined plasma current density profile. The equilibrium shape is defined in terms of constraints on locations of X/O points or on values of psi at certain locations.

In this example, we will show how JAX's AD capabilities can be used to solve the inverse problem using gradient-based methods.

### Initial set-up

Initially, we set the machine and initialise the solver as standard.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)

In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# initialise the limiter geometry
from freegsnke.jaxify import limiter_func
jLimiter = limiter_func.Limiter_handler(eq, eq.tokamak.limiter)

# initialise the profiles
from freegsnke.jaxify.jtor import JConstrainPaxisIp, JLao85
jProfile = JConstrainPaxisIp(
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

# initialise the static solver
from freegsnke.jaxify import GSstaticsolver
jGS = GSstaticsolver.NKGSsolver(eq, jProfile, jLimiter)
jProfilePars = jProfile.init_params

### Inverse problem formulation

In the inverse problem, we wish to find the coil currents that give a desired equilibrium shape. We can write this in terms of a constrained optimization problem. We define a cost function $\mathcal{J}(I_j,\psi)$ that quantifies how close we are to our desired shape - this desired shape is represented by a set of constraints eg location of X-point, isofluxes, psi values at certain locations etc. Let us begin by defining the plasma core boundary with 4 points.. 2 x-points and 2 points on the outboard and inboard mid-plane. 

In [ ]:
Rx = 0.6      # X-point radius
Zx = 1.1      # X-point height
Ra = .85
Rout = 1.4    # outboard midplane radius
Rin = 0.34    # inboard midplane radius

# Array of boundary points
xb = jnp.array([[Rx, Rx, Rin, Rout], [Zx, -Zx, 0.,0.]])

We need to set up the vector of control currents because the machine description may contain passive structures or other coils that are not to be used for control. For example, in this case, the machine has 150 coils, but the majority of these are passive structures that we want to ignore. In this example, we also set the solenoid current to be fixed and.

In [ ]:
# Set solenoid to be fixed
eq.tokamak.set_coil_current('Solenoid', 5000)
eq.tokamak['Solenoid'].control = False  # ensures the current in the Solenoid is fixed

# get full initial vector of currents
init_currs=jnp.asarray(eq.tokamak.getCurrentsVec())

# List of control coils from eq.tokamak object
control_coils = [
            (label, coil) for label, coil in eq.tokamak.coils if coil.control
        ]

# Masking matrix that says which coils are control
control_mask = jnp.array(
            [coil.control for label, coil in eq.tokamak.coils]
        ).astype(bool)

# Control matrix
Rmat = jnp.eye(control_mask.shape[0])[control_mask]

First, let's get an initial guess for the currents using a linear approximation. We assume that the plasma component $\psi_p$ is constant, and find the currents that generate a metal component $\psi_m$ that approximates the target shape.

In [ ]:
# set desired null_points locations
# this can include X-point and O-point locations
null_points = [[Rx, Rx], [Zx, -Zx]]

# set desired isoflux constraints with format 
# isoflux_set = [isoflux_0, isoflux_1 ... ] 
# with each isoflux_i = [R_coords, Z_coords]
# isoflux_set = np.array([[[Rx, Rx, Rin, Rout, 1.3, 1.3, .8,.8], [Zx, -Zx, 0.,0., 2.1, -2.1,1.62,-1.62]]])
isoflux_set = np.array([[[Rx, Rx, Rin, Rout], [Zx, -Zx, 0.,0.]]])

# instantiate the freegsnke constrain object
from freegsnke.inverse import Inverse_optimizer
constrain = Inverse_optimizer(null_points=null_points,
                              isoflux_set=isoflux_set)

# Try freegsnke solve
# initialise the profiles
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-6,
                     target_relative_psit_update=1e-3,
					 max_solving_iterations=7,
                     verbose=True, # print output
                     l2_reg=np.array([1e-12]*10+[1e-6]), 
                     )
jCurr0 = eq.tokamak.getCurrentsVec()
psi0 = eq.psi()
inverse_current_values = eq.tokamak.getCurrents()

# save coil currents to file
import pickle
with open('data/test_diverted_currents_PaxisIp.pk', 'wb') as f:
    pickle.dump(obj=inverse_current_values, file=f)

In [ ]:
# Solve the forward GS problem using this set of coil currents
psi0 = eq.psi()
psi = jGS.solve(init_psi=psi0,
                    profilePars = jProfilePars,
                    currentvec = jCurr0,
                    target_relative_tolerance = 1e-6,
                    use_newton = False,
                    verbose=True, # print output
                    )
import matplotlib.pyplot as plt

# Plot the differences
# fig1, ax1 = plt.subplots(1, 1, figsize=(5, 8), dpi=80)
# plt.contour(eq.R,eq.Z,psi,100)
# eq.tokamak.plot(axis=ax1, show=False)
# plt.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, 'k', 3.0)
# ax1.set_xlim(0.1, 2.15)
# ax1.set_ylim(-2.25, 2.25)
# plt.tight_layout()
# plt.colorbar(); plt.show()


We will consider a fully nonlinear cost function - that the normalised poloidal flux, $\psi_n = \frac{\psi - \psi_o}{\psi_x - \psi_o}$ should be 1 at these boundary points.

$$\mathcal{J} = \frac{1}{2}\sum_p^{N_p} \left( \psi_n(\psi(r_p,z_p))-1 \right) .^2 $$

Let's now write a function calculate this cost function using our JAX solver.

In [ ]:
# Define cost function in terms of the control currents and the contrain locations
# We pass an initial guess for psi for the forward solve
# Here, the profileparams are constant and we don't need gradients wrt them
# so we can leave them out of the function arguments
def Jcost(Ic, xp, old_psi):
	
	Iscale = 1e4
    # Get full current vector from subset of control currents
	jCurr = init_currs + Iscale * (Rmat.T @ Ic)

    # Solve the forward GS problem using this set of coil currents
	psi = jGS.solve(init_psi=old_psi,
                    profilePars = jProfilePars,
                    currentvec = jCurr,
                    target_relative_tolerance = 1e-8,
                    use_newton = False,
                    verbose=False, # print output
                    )
	
	# Set up an interpolation to get the values of psi at the control points
	import interpax as ix
	r1d = jGS.R[:,0]
	z1d = jGS.Z[0,:]
	f_psi = ix.Interpolator2D(r1d,z1d,psi)
	psi_p = f_psi(xp[0,:],xp[1,:])

	# Get x and o-points
	opts,xpts = jGS.critpoints(psi)
	psi_o = opts[0,2]
	psi_x = xpts[0,2]

	# If no x-point, ignore limiter for now and set to max psi
	psi_b = jnp.where(psi_x<0,jnp.amax(psi),psi_x)

	# Calculate the normalised psi values at these points
	psi_n = (psi_p - psi_o)/(psi_b - psi_o)

	# Calculate the cost function in terms of the error
	wj=jnp.array([1e-12]*10 + [1e-6])
	jcost = 0.5*jnp.sum((psi_n-1.0)**2.0) + 0.5*jnp.dot(wj,Ic)

	return jcost, psi

Now let's run a few iterations of simple gradient descent to see if it works.

In [ ]:
# let's test it out once to see if it works
I0 = 1/1e4 * (Rmat @ jCurr0)
print(I0)
J0, psi0 = Jcost(I0, xb, eq.psi())

print("Initial Jcost=",J0)
print("Initial currents=",I0)
Icontrol = I0
psi_old = psi0
stepsize=0.001
for _ in range(5):
	value, grad = jax.value_and_grad(Jcost,argnums=0,has_aux=True)(Icontrol, xb, psi_old)
	print("Jcost=",value[0])
	print("Gradient Norm=",jnp.linalg.norm(grad))
	Icontrol = Icontrol - stepsize*grad
	psi_old = value[1]

The gradient descent algorithm works smoothly, but we need a good idea of stepsize to enable it to work efficiently. A better idea might be to use the Newton method - this requires the second-derivative, which can be obtained using JAX too. However, we found that this is numerically unstable because it is essentially calculating a second-derivative of an iterative Krylov solver. An easy alternative is to use ready-made libraries on the Python/JAX ecosystem to perform the optimization. 

In [ ]:
import jaxopt

print("Initial Jcost=",J0)
print("Initial currents=",I0)
Icontrol = I0
psi_old = psi0

solver = jaxopt.GradientDescent(fun=Jcost, has_aux=True, stepsize=0.005, maxls=4, maxiter=10, tol=1e-2, jit=False, verbose=True)
res = solver.run(init_params=Icontrol, xp=xb, old_psi=psi_old)
print("Final Jcost=",Jcost(res[0],xp=xb,old_psi=psi_old)[0])
print("Final currents=",res[0])

We can also interface with the scipy.minimize function. In this case, the backtracking line-search initialises from a value from 1.0, so several reductions are needed to reach the optimal step size of 0.005.

In [ ]:
from scipy.optimize import minimize

print("Initial Jcost=",J0)
print("Initial currents=",I0)
Icontrol = I0
psi_old = psi0

def f(x):
	return Jcost(x, xb, psi_old)[0]

def dfdx(x):
	return jax.grad(f)(x)

result = minimize(f, x0=Icontrol, method='BFGS', jac=dfdx, options={'disp': True, 'gtol': 1e-1})

